In [2]:
from pathlib import Path
import pandas as pd

# ==================================================
# 0. 데이터 로드 및 날짜 자동 변환
# ==================================================
data_dir = Path('..') / 'data' / 'raw'

customers = pd.read_csv(data_dir / 'customers.csv')
orders = pd.read_csv(data_dir / 'orders.csv')
order_items = pd.read_csv(data_dir / 'order_items.csv')
products = pd.read_csv(data_dir / 'products.csv')

def convert_date_columns(dfs_dict):
    for name, df in dfs_dict.items():
        date_cols = [col for col in df.columns if 'date' in col.lower()]
        if date_cols:
            for col in date_cols:
                df[col] = pd.to_datetime(df[col])
    return dfs_dict

dfs = convert_date_columns({
    'customers': customers, 'orders': orders,
    'order_items': order_items, 'products': products
})


# ==================================================
# 1. 주문 상세 금액 계산 (quantity * unit_price)
# ==================================================
def add_item_sales(order_items: pd.DataFrame) -> pd.DataFrame:
    """주문 상세별 판매금액(quantity * unit_price)을 계산합니다."""
    result = order_items.copy()
    result["sales_amount"] = result["quantity"] * result["unit_price"]
    return result

# 1번 실행 및 확인
order_items_calc = add_item_sales(order_items)
print("=== 1. 주문 상세 금액 계산 결과 (상위 3건) ===")
display(order_items_calc.head(3))


# ==================================================
# 2. 주문과 주문 상세 병합
# ==================================================
def merge_orders_and_items(orders: pd.DataFrame, order_items_df: pd.DataFrame) -> pd.DataFrame:
    """주문(orders)과 금액이 계산된 주문 상세(order_items)를 병합합니다."""
    return orders.merge(order_items_df, on='order_id', how='inner')

# 2번 실행 및 확인
orders_items_df = merge_orders_and_items(orders, order_items_calc)
print("\n=== 2. 주문 + 주문 상세 병합 결과 (상위 3건) ===")
display(orders_items_df.head(3))


# ==================================================
# 3. 상품 데이터 병합
# ==================================================
def merge_products(orders_items_df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    """주문+주문상세 데이터에 상품 정보(products)를 병합합니다."""
    return orders_items_df.merge(products, on='product_id', how='inner')

# 3번 실행 및 확인 (최종 병합 데이터 생성)
full_df = merge_products(orders_items_df, products)
print("\n=== 3. 상품 데이터까지 병합된 전체 데이터 (상위 3건) ===")
display(full_df.head(3))


# ==================================================
# 4. 카테고리별 집계
# ==================================================
def summarize_by_category(full_df: pd.DataFrame) -> pd.DataFrame:
    """카테고리별 판매수량, 총 매출액, 고유 주문 건수를 집계합니다."""
    return (
        full_df.groupby('category', as_index=False)
        .agg(
            total_quantity=('quantity', 'sum'),
            total_sales=('sales_amount', 'sum'),
            order_count=('order_id', 'nunique')
        )
        .sort_values('total_sales', ascending=False)
    )

# 4번 실행 및 확인
category_summary = summarize_by_category(full_df)
print("\n=== 4. 카테고리별 집계 ===")
display(category_summary)


# ==================================================
# 5. 월별 집계
# ==================================================
def summarize_by_month(full_df: pd.DataFrame) -> pd.DataFrame:
    """월별(YYYY-MM) 판매수량, 총 매출액, 고유 주문 건수를 집계합니다."""
    df = full_df.copy()
    df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
    
    return (
        df.groupby('year_month', as_index=False)
        .agg(
            total_quantity=('quantity', 'sum'),
            total_sales=('sales_amount', 'sum'),
            order_count=('order_id', 'nunique')
        )
        .sort_values('year_month')
    )

# 5번 실행 및 확인
monthly_summary = summarize_by_month(full_df)
print("\n=== 5. 월별 집계 (상위 5개월) ===")
display(monthly_summary.head(5))


# ==================================================
# 6. 핵심 지표 함수 (KPI)
# ==================================================
def calculate_key_metrics(full_df: pd.DataFrame) -> pd.DataFrame:
    """전체 데이터에 대한 핵심 경영/매출 지표(KPI)를 산출합니다."""
    total_sales = full_df['sales_amount'].sum()
    total_orders = full_df['order_id'].nunique()
    total_quantity = full_df['quantity'].sum()
    total_customers = full_df['customer_id'].nunique()
    
    aov = total_sales / total_orders if total_orders > 0 else 0
    arpu = total_sales / total_customers if total_customers > 0 else 0

    return pd.DataFrame({
        '총 매출액': [f"{total_sales:,}원"],
        '총 주문건수': [f"{total_orders:,}건"],
        '총 판매수량': [f"{total_quantity:,}개"],
        '구매 고객수': [f"{total_customers:,}명"],
        '평균 주문금액(AOV)': [f"{round(aov):,}원"],
        '고객당 평균구매액': [f"{round(arpu):,}원"]
    })

# 6번 실행 및 확인
kpi_summary = calculate_key_metrics(full_df)
print("\n=== 6. 핵심 지표(KPI) ===")
display(kpi_summary)

=== 1. 주문 상세 금액 계산 결과 (상위 3건) ===


,order_item_id,order_id,product_id,quantity,unit_price,sales_amount
0,1,1,216,4,13600,54400
1,2,1,34,1,87000,87000
2,3,1,187,1,93000,93000



=== 2. 주문 + 주문 상세 병합 결과 (상위 3건) ===


,order_id,customer_id,order_date,payment_method,order_status,order_item_id,product_id,quantity,unit_price,sales_amount
0,1,1121,2026-05-18,간편결제,배송중,1,216,4,13600,54400
1,1,1121,2026-05-18,간편결제,배송중,2,34,1,87000,87000
2,1,1121,2026-05-18,간편결제,배송중,3,187,1,93000,93000



=== 3. 상품 데이터까지 병합된 전체 데이터 (상위 3건) ===


...

,category,total_quantity,total_sales,order_count
4,생활가전,2205,457423100,1263
8,패션,3254,313067200,1776
7,전자기기,3112,286750100,1705
9,홈인테리어,1963,173982200,1090
5,스포츠,1956,154676700,1117
3,뷰티,2929,135823500,1555
6,식품,2976,116157900,1593
2,반려동물,1726,110816200,987
1,문구,1635,46005800,951
0,도서,1840,43071600,1075



=== 5. 월별 집계 (상위 5개월) ===


,year_month,total_quantity,total_sales,order_count
0,2025-01,1147,87322800,296
1,2025-02,878,68504700,233
2,2025-03,1193,85551000,312
3,2025-04,1369,104790900,356
4,2025-05,1314,105969700,329



=== 6. 핵심 지표(KPI) ===


,총 매출액,총 주문건수,총 판매수량,구매 고객수,평균 주문금액(AOV),고객당 평균구매액
0,"1,837,774,300원","6,000건","23,596개","1,102명","306,296원","1,667,672원"
